In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'


In [13]:
import os
import json
import glob
from safetensors import safe_open
from safetensors.torch import load_file

def load_chunked_safetensors(model_dir, tensor_keys=None, verbose=False):
    """
    Load tensors from multiple safetensors chunks into a single dictionary.
    
    Args:
        model_dir: Directory containing the safetensors chunks or a direct path to a .safetensors file
        tensor_keys: Optional list of specific tensor keys to load (for memory efficiency)
        verbose: If True, print debugging information during loading
    
    Returns:
        Dictionary of tensor name to tensor
    """
    # Check if model_dir is a file or directory
    if os.path.isfile(model_dir) and model_dir.endswith('.safetensors'):
        # If model_dir is a direct path to a .safetensors file
        if verbose:
            print(f"Loading single safetensors file: {model_dir}")
        return load_file(model_dir) if tensor_keys is None else {
            k: v for k, v in load_file(model_dir).items() if k in tensor_keys
        }
    
    # Find all safetensors files in the directory
    chunk_files = sorted(glob.glob(os.path.join(model_dir, "*.safetensors")))
    if not chunk_files:
        raise FileNotFoundError(f"No safetensors files found in {model_dir}")
    
    if verbose:
        print(f"Found {len(chunk_files)} safetensors files in {model_dir}")
        for i, f in enumerate(chunk_files):
            print(f"  {i+1}. {os.path.basename(f)}")
    
    # Look for index file
    index_file = os.path.join(model_dir, "model.safetensors.index.json")
    index_map = {}
    
    if os.path.exists(index_file):
        # Load the index to find which chunk contains which tensor
        with open(index_file, 'r') as f:
            index_data = json.load(f)
        
        if verbose:
            print(f"Found index file with {len(index_data['weight_map'])} tensor mappings")
        
        # Build mapping from tensor name to file chunk
        for tensor_name, metadata in index_data['weight_map'].items():
            if tensor_keys is None or tensor_name in tensor_keys:
                index_map[tensor_name] = metadata
    
    # Load tensors from chunks
    all_tensors = {}
    
    if index_map:
        # Group tensors by file to minimize file operations
        tensors_by_file = {}
        for tensor_name, file_name in index_map.items():
            if file_name not in tensors_by_file:
                tensors_by_file[file_name] = []
            tensors_by_file[file_name].append(tensor_name)
            
        # Load tensors from each file
        for file_idx, (file_name, tensor_names) in enumerate(tensors_by_file.items()):
            file_path = os.path.join(model_dir, file_name)
            if verbose:
                print(f"Loading {len(tensor_names)} tensors from {file_name} ({file_idx+1}/{len(tensors_by_file)})")
            
            # Use context manager instead of explicit close
            with safe_open(file_path, framework="pt") as f:
                for tensor_idx, tensor_name in enumerate(tensor_names):
                    if verbose and tensor_idx % 100 == 0 and tensor_idx > 0:
                        print(f"  Loaded {tensor_idx}/{len(tensor_names)} tensors...")
                    all_tensors[tensor_name] = f.get_tensor(tensor_name)
    else:
        # Without an index, load all chunks and merge
        for chunk_idx, chunk_file in enumerate(chunk_files):
            if verbose:
                print(f"Loading chunk {chunk_idx+1}/{len(chunk_files)}: {os.path.basename(chunk_file)}")
            
            tensors = load_file(chunk_file)
            
            if verbose:
                print(f"  Loaded {len(tensors)} tensors from chunk")
            
            # Filter if specific keys were requested
            if tensor_keys:
                tensors = {k: v for k, v in tensors.items() if k in tensor_keys}
                if verbose and len(tensors) < len(tensor_keys):
                    print(f"  Filtered to {len(tensors)} requested tensors")
            
            all_tensors.update(tensors)
    
    if verbose:
        print(f"Successfully loaded {len(all_tensors)} total tensors")
    
    return all_tensors



In [14]:
path1 = "/data/jamesliu/sglang/phoenix_1layer_lora/base_model"
path2 = "/home/jamesliu/.cache/huggingface/hub/models--meta-llama--Meta-Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659"



tensors1 = load_chunked_safetensors(path1)
tensors2 = load_chunked_safetensors(path2)




In [15]:
print(tensors1.keys())
print(tensors2.keys())

dict_keys(['lm_head.weight', 'model.layers.30.input_layernorm.weight', 'model.layers.30.mlp.down_proj.base_layer.weight', 'model.layers.30.mlp.down_proj.lora_A.weight', 'model.layers.30.mlp.down_proj.lora_B.weight', 'model.layers.30.post_attention_layernorm.weight', 'model.layers.31.input_layernorm.weight', 'model.layers.31.mlp.down_proj.base_layer.weight', 'model.layers.31.mlp.down_proj.lora_A.weight', 'model.layers.31.mlp.down_proj.lora_B.weight', 'model.layers.31.mlp.gate_proj.base_layer.weight', 'model.layers.31.mlp.gate_proj.lora_A.weight', 'model.layers.31.mlp.gate_proj.lora_B.weight', 'model.layers.31.mlp.up_proj.base_layer.weight', 'model.layers.31.mlp.up_proj.lora_A.weight', 'model.layers.31.mlp.up_proj.lora_B.weight', 'model.layers.31.post_attention_layernorm.weight', 'model.layers.31.self_attn.k_proj.base_layer.weight', 'model.layers.31.self_attn.k_proj.lora_A.weight', 'model.layers.31.self_attn.k_proj.lora_B.weight', 'model.layers.31.self_attn.o_proj.base_layer.weight', 'mo

In [19]:
index1 = "model.layers.30.self_attn.q_proj.base_layer.weight"
index2 = "model.layers.30.self_attn.q_proj.weight"

print(index1 in tensors1.keys())
print(index2 in tensors2.keys())

T1 = tensors1[index1]
T2 = tensors2[index2]

print(T1.shape)
print(T2.shape)

print(T1)
print(T2)


True
True
torch.Size([4096, 4096])
torch.Size([4096, 4096])
tensor([[ 0.0181,  0.0042,  0.0225,  ...,  0.0089, -0.0126, -0.0077],
        [ 0.0173, -0.0576,  0.0239,  ...,  0.0008,  0.0322,  0.0140],
        [-0.0226, -0.0249, -0.0217,  ..., -0.0043,  0.0087, -0.0013],
        ...,
        [ 0.0361,  0.0203, -0.0371,  ..., -0.0295, -0.0199,  0.0532],
        [-0.0366, -0.0183,  0.0150,  ..., -0.0023, -0.0129,  0.0094],
        [-0.0215, -0.0143,  0.0087,  ..., -0.0267, -0.0098, -0.0210]],
       dtype=torch.bfloat16)
tensor([[ 0.0181,  0.0042,  0.0225,  ...,  0.0089, -0.0126, -0.0077],
        [ 0.0173, -0.0576,  0.0239,  ...,  0.0008,  0.0322,  0.0140],
        [-0.0226, -0.0249, -0.0217,  ..., -0.0043,  0.0087, -0.0013],
        ...,
        [ 0.0361,  0.0203, -0.0371,  ..., -0.0295, -0.0199,  0.0532],
        [-0.0366, -0.0183,  0.0150,  ..., -0.0023, -0.0129,  0.0094],
        [-0.0215, -0.0143,  0.0087,  ..., -0.0267, -0.0098, -0.0210]],
       dtype=torch.bfloat16)
